## Unsupervised Learning
#### by **Ivan Alducin**
<p><img src="https://cdn.gritglobal.io/uploads/too-broad-customer-segmentation.jpg" width="1000"/></p>

## Segmentacion de Clientes
<p>En esté capitulo nos vamos a enfocar en entender y trabajar un caso de uso para segmentación de clientes, pero antes de eso aquí una pequeña lista de más aplicaciones que se pueden trabajar con los datos recopliados de mis clientes

- Estadística Descriptiva
- Segmentación de Clientes
- Predicción de Abandono
- Valor del Cliente a traves del tiempo (CTLV)

La segmentación la vamos a hacer con base en una metodolgía llamada <b>RFM</b>

</p>

In [ ]:
# Importa Pandas, Numpy, Seaborn y Matplotlib
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Importa el archivo "Online Retail.csv"
df = pd.read_csv("M30 Online Retail.csv", encoding="latin1")
df["INVOICE_DATE"] = pd.to_datetime(df["INVOICE_DATE"], dayfirst=True, errors="coerce")
df = df.dropna(subset=["CUSTOMER_ID", "INVOICE_DATE"]).copy()

df.head()

In [ ]:
# Análisis Exploratorio
print("Dimensiones del dataset:", df.shape)
print("\nTipos de datos:")
print(df.dtypes)

print("\nValores nulos:")
print(df.isnull().sum())

print("\nResumen estadístico:")
print(df.describe(include="all"))

df.head()

## Recency
<p>Indicador que nos dice que tan reciente es la compra de un cliente</p>

In [ ]:
# Obtener los clientes unicos
customer = pd.DataFrame(df["CUSTOMER_ID"].dropna().unique(), columns=["CUSTOMER_ID"])
customer.head()

In [ ]:
# Obtener la última fecha de compra por cliente
max_purchase = df.groupby("CUSTOMER_ID")["INVOICE_DATE"].max().reset_index()
max_purchase.head()

In [ ]:
# Vamos a calcular nuestra metrica de Recency, esto lo haremos restando los días de la última fecha de compra a cada observacón
max_purchase["RECENCY"] = (df["INVOICE_DATE"].max() - max_purchase["INVOICE_DATE"]).dt.days
max_purchase.head()

In [ ]:
# Unir el DataFrame de clientes únicos con el que acabamos de crear de la última fecha de compra
customer = pd.merge(customer, max_purchase[["CUSTOMER_ID", "RECENCY"]], on="CUSTOMER_ID")
customer.head()

In [ ]:
# Grafica un histograma de Recency
plt.figure(figsize=(10,6))
sns.histplot(customer["RECENCY"], bins=30, kde=True)
plt.title("Distribución de Recency")
plt.xlabel("Recency")
plt.ylabel("Frecuencia")
plt.show()

In [ ]:
# Imprime la Estadística de Resumen para Recency
customer["RECENCY"].describe()

## Frequency
<p>Frecuencia con la que un cliente compra uno o más productos</p>

In [ ]:
# Obtener el número de compras por cliente
frequency = df.groupby("CUSTOMER_ID")["INVOICE_NO"].nunique().reset_index()
frequency.columns = ["CUSTOMER_ID", "FREQUENCY"]
frequency.head()

In [ ]:
# Unir el DataFrame que acabamos de crear con el de los clientes unicos
customer = pd.merge(customer, frequency, on="CUSTOMER_ID")
customer.head()

In [ ]:
# Grafica un histograma de Frequency
plt.figure(figsize=(10,6))
sns.histplot(customer["FREQUENCY"], bins=30, kde=True)
plt.title("Distribución de Frequency")
plt.xlabel("Frequency")
plt.ylabel("Frecuencia")
plt.show()

In [ ]:
# Imprime la Estadística de Resumen para Frequency
customer["FREQUENCY"].describe()

## Monetary
<p>Valor del monto total que ha gastado un cliente en la compra de mis productos</p>

In [ ]:
# Calcular el monto total por cada compra
df["MONETARY"] = df["QUANTITY"] * df["UNIT_PRICE"]

# Obtener el valor monetario de compra por cliente
monetary = df.groupby("CUSTOMER_ID")["MONETARY"].sum().reset_index()
monetary.head()

In [ ]:
# Unir el DataFrame que acabamos de crear con el de los clientes unicos
customer = pd.merge(customer, monetary, on="CUSTOMER_ID")
customer.head()

In [ ]:
# Grafica un histograma de Monetary
plt.figure(figsize=(10,6))
sns.histplot(customer["MONETARY"], bins=30, kde=True)
plt.title("Distribución de Monetary")
plt.xlabel("Monetary")
plt.ylabel("Frecuencia")
plt.show()

In [ ]:
# Imprime la Estadística de Resumen para Monetary
customer["MONETARY"].describe()

## Algoritmo k-Means
<p>Ya creamos nuestros indicadores principales de la metodología RFM. es hora de hacer <i>Machine Learning</i>. Para ello utilizaremos un algoritmo no supervisado llamado <b>k-Means</b></p>
<p><img src="https://miro.medium.com/max/818/1*fG8u8nV7qR91wDyFDEEV-g.png" width="250"/></p>

In [ ]:
# Funcion para ordenar los clusters
def order_cluster(cluster_field_name, target_field_name, df, ascending):
    new_cluster_field_name = 'new_' + cluster_field_name
    df_new = df.groupby(cluster_field_name)[target_field_name].mean().reset_index()
    df_new = df_new.sort_values(by=target_field_name,ascending=ascending).reset_index(drop=True)
    df_new['index'] = df_new.index
    df_final = pd.merge(df,df_new[[cluster_field_name,'index']], on=cluster_field_name)
    df_final = df_final.drop([cluster_field_name],axis=1)
    df_final = df_final.rename(columns={"index":cluster_field_name})
    return df_final

## Elbow Method
<p>¿Cual es mi número óptimo de clusters? Vamos a contruir una <i>gráfica de codo</i> para averiguarlo</p>

In [ ]:
# Importa la librería de kMeans
from sklearn.cluster import KMeans

In [ ]:
# Configuración inicial - Vamos a tomar como referencia el indicador de Recency
sse = {}
recency = customer[["RECENCY"]].copy()

for k in range(1, 10):
    # Instancia el algoritmo de k-means iterando sobre k
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=1, max_iter=50)
    
    # Entrena el algoritmo
    kmeans.fit(recency)
    
    # Adjunta las etiquetas
    recency["clusters"] = kmeans.labels_
    
    # Adunta la inercia o variación al arreglo sse
    sse[k] = kmeans.inertia_
    
# Grafico de codo (Elbow)
plt.figure(figsize=(12,8))
plt.plot(list(sse.keys()), list(sse.values()))
plt.xlabel("Número de Clusters")
plt.ylabel("SSE")
plt.title("Elbow Method para Recency")
plt.show()

In [ ]:
# Instanciar el algoritmo con 4 clusters para Recency 
kmeans = KMeans(n_clusters=4, random_state=42, n_init=1, max_iter=50)

# Entrenar el algoritmo
kmeans.fit(customer[["RECENCY"]])

# Obtener las predicciones
customer["RECENCY_CLUSTER"] = kmeans.predict(customer[["RECENCY"]])

# Ordenar los clusters
customer = order_cluster("RECENCY_CLUSTER", "RECENCY", customer, False)

# Estadística Descriptiva del cluster creado
customer.groupby("RECENCY_CLUSTER")["RECENCY"].describe()

In [ ]:
# Instanciar el algoritmo con 4 clusters para Frequency 
kmeans = KMeans(n_clusters=4, random_state=42, n_init=1, max_iter=50)

# Entrenar el algoritmo
kmeans.fit(customer[["FREQUENCY"]])

# Obtener las predicciones
customer["FREQUENCY_CLUSTER"] = kmeans.predict(customer[["FREQUENCY"]])

# Ordenar los clusters
customer = order_cluster("FREQUENCY_CLUSTER", "FREQUENCY", customer, True)

# Estadística Descriptiva de los clusters
customer.groupby("FREQUENCY_CLUSTER")["FREQUENCY"].describe()

In [ ]:
# Instanciar el algoritmo con 4 clusters para Monetary 
kmeans = KMeans(n_clusters=4, random_state=42, n_init=1, max_iter=50)

# Entrenar el algoritmo
kmeans.fit(customer[["MONETARY"]])

# Obtener las predicciones
customer["MONETARY_CLUSTER"] = kmeans.predict(customer[["MONETARY"]])

# Ordenar los clusters ¿Como tienes que ordenar el cluster?
customer = order_cluster("MONETARY_CLUSTER", "MONETARY", customer, True)

# Estadística Descriptiva de los clusters
customer.groupby("MONETARY_CLUSTER")["MONETARY"].describe()

## Score de Segmentación
<p>El algoritmo de k-means nos da una segmentación generalizada, pero podemos personalizarla aún más creando una métrica que asigne una calificación al valor del cluster. Esto es lo que vamos a hacer!!</p>

In [ ]:
# Vamos a crear nuestro score sumando el valor de cada uno de los clusters
customer["SCORE"] = customer["RECENCY_CLUSTER"] + customer["FREQUENCY_CLUSTER"] + customer["MONETARY_CLUSTER"]

# Obtener el promedio para cada una de las métricas de las calificaciones creadas (Score)
customer.groupby("SCORE")[["RECENCY", "FREQUENCY", "MONETARY"]].mean()

In [ ]:
# Crea una funcion que asigne lo siguiente: 
# Si score <= 1 entonces 'Low-Value', si score >1 y <=4 entonces 'Average', si score >4 y <=6 entonces 'Potential', por último si score >6 entonces 'High-Value'
def segment(score):
    if score <= 1:
        return "Low-Value"
    elif score > 1 and score <= 4:
        return "Average"
    elif score > 4 and score <= 6:
        return "Potential"
    else:
        return "High-Value"

# Crear una columna aplicando esta función al campo 'SCORE'
customer["SEGMENT"] = customer["SCORE"].apply(segment)
customer.head()

In [ ]:
# Vamos a dar un vistazo a la tabla final
customer.head()

In [ ]:
# Imprime la proporción o el total de clientes por segmento
customer["SEGMENT"].value_counts()

In [ ]:
# Define un estilo 'bmh'
plt.style.use("bmh")

# Filtra los valores para RECENCY < 4000
customer_graph = customer[customer["RECENCY"] < 4000]

# Crea un grafico de dispersion de 'MONETARY' VS 'RECENCY' por Segmento
plt.figure(figsize=(12,8))
sns.scatterplot(
    data=customer_graph,
    x="RECENCY",
    y="MONETARY",
    hue="SEGMENT"
)
plt.title("MONETARY vs RECENCY por Segmento")
plt.show()

In [ ]:
# Crea un grafico de dispersion de 'MONETARY' vs 'FREQUENCY' vs  por Segmento
plt.figure(figsize=(12,8))
sns.scatterplot(
    data=customer_graph,
    x="FREQUENCY",
    y="MONETARY",
    hue="SEGMENT"
)
plt.title("MONETARY vs FREQUENCY por Segmento")
plt.show()